# Hash Function Comparisons

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import hashlib
import time
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import seaborn as sns

class HashComparator:
    def __init__(self):
        self.algorithms = ['md5', 'sha1', 'sha224', 'sha256', 'sha384', 'sha512', 'blake2b', 'blake2s']
        self.results = {}

    def hash_text(self, text, algorithm):
        if algorithm == 'blake2b':
            return hashlib.blake2b(text.encode()).hexdigest()
        elif algorithm == 'blake2s':
            return hashlib.blake2s(text.encode()).hexdigest()
        else:
            return hashlib.new(algorithm, text.encode()).hexdigest()

    def compare_hash_lengths(self):
        test_string = "cybersecurity"
        lengths = {}

        for algo in self.algorithms:
            hash_value = self.hash_text(test_string, algo)
            lengths[algo] = len(hash_value)

        return lengths

    def measure_performance(self, data_size=1000000):
        test_data = "a" * data_size
        performance = {}

        for algo in self.algorithms:
            start_time = time.time()
            for _ in range(100):
                self.hash_text(test_data[:1000], algo)
            end_time = time.time()
            performance[algo] = (end_time - start_time) / 100

        return performance

    def analyze_distribution(self, sample_size=1000):
        distributions = {}

        for algo in self.algorithms:
            hex_chars = []
            for i in range(sample_size):
                hash_val = self.hash_text(f"test{i}", algo)
                hex_chars.extend(list(hash_val))

            char_counts = Counter(hex_chars)
            distributions[algo] = char_counts

        return distributions

    def avalanche_effect(self, base_string="cybersecurity"):
        avalanche_data = {}

        for algo in self.algorithms:
            original_hash = self.hash_text(base_string, algo)
            bit_differences = []

            for i in range(len(base_string)):
                modified_string = base_string[:i] + chr(ord(base_string[i]) ^ 1) + base_string[i+1:]
                modified_hash = self.hash_text(modified_string, algo)

                diff_count = sum(c1 != c2 for c1, c2 in zip(original_hash, modified_hash))
                bit_differences.append(diff_count)

            avalanche_data[algo] = np.mean(bit_differences)

        return avalanche_data

    def collision_resistance_test(self, iterations=10000):
        collision_data = {}

        for algo in self.algorithms:
            hashes = set()
            collisions = 0

            for i in range(iterations):
                hash_val = self.hash_text(f"collision_test_{i}_{np.random.randint(0, 1000000)}", algo)
                if hash_val in hashes:
                    collisions += 1
                hashes.add(hash_val)

            collision_data[algo] = collisions

        return collision_data

    def security_analysis(self):
        security_ratings = {
            'md5': {'security': 'Broken', 'score': 1, 'notes': 'Cryptographically broken'},
            'sha1': {'security': 'Deprecated', 'score': 2, 'notes': 'Collision attacks found'},
            'sha224': {'security': 'Secure', 'score': 4, 'notes': 'Truncated SHA-256'},
            'sha256': {'security': 'Secure', 'score': 5, 'notes': 'Industry standard'},
            'sha384': {'security': 'Secure', 'score': 5, 'notes': 'Truncated SHA-512'},
            'sha512': {'security': 'Secure', 'score': 5, 'notes': 'High security'},
            'blake2b': {'security': 'Secure', 'score': 5, 'notes': 'Modern, fast'},
            'blake2s': {'security': 'Secure', 'score': 5, 'notes': 'Modern, optimized'}
        }
        return security_ratings

    def run_comprehensive_analysis(self):
        print("=== HASH FUNCTION COMPARISON ANALYSIS ===\n")

        print("1. HASH OUTPUT LENGTHS")
        lengths = self.compare_hash_lengths()
        for algo, length in lengths.items():
            print(f"{algo.upper()}: {length} characters")

        print("\n2. PERFORMANCE ANALYSIS")
        performance = self.measure_performance()
        sorted_perf = sorted(performance.items(), key=lambda x: x[1])
        for algo, time_taken in sorted_perf:
            print(f"{algo.upper()}: {time_taken:.6f} seconds")

        print("\n3. AVALANCHE EFFECT")
        avalanche = self.avalanche_effect()
        for algo, avg_diff in avalanche.items():
            print(f"{algo.upper()}: {avg_diff:.2f} character differences on average")

        print("\n4. COLLISION RESISTANCE TEST")
        collisions = self.collision_resistance_test()
        for algo, collision_count in collisions.items():
            print(f"{algo.upper()}: {collision_count} collisions found")

        print("\n5. SECURITY ASSESSMENT")
        security = self.security_analysis()
        for algo, data in security.items():
            print(f"{algo.upper()}: {data['security']} - {data['notes']}")

        self.visualize_results(lengths, performance, avalanche, security)

    def visualize_results(self, lengths, performance, avalanche, security):
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

        algos = list(lengths.keys())
        colors = plt.cm.Set3(np.linspace(0, 1, len(algos)))

        ax1.bar(algos, lengths.values(), color=colors)
        ax1.set_title('Hash Output Lengths')
        ax1.set_ylabel('Characters')
        ax1.tick_params(axis='x', rotation=45)

        perf_values = [performance[algo] for algo in algos]
        ax2.bar(algos, perf_values, color=colors)
        ax2.set_title('Performance (Lower is Better)')
        ax2.set_ylabel('Time (seconds)')
        ax2.tick_params(axis='x', rotation=45)

        avalanche_values = [avalanche[algo] for algo in algos]
        ax3.bar(algos, avalanche_values, color=colors)
        ax3.set_title('Avalanche Effect')
        ax3.set_ylabel('Average Character Differences')
        ax3.tick_params(axis='x', rotation=45)

        security_scores = [security[algo]['score'] for algo in algos]
        bars = ax4.bar(algos, security_scores, color=colors)
        ax4.set_title('Security Rating')
        ax4.set_ylabel('Security Score (1-5)')
        ax4.set_ylim(0, 6)
        ax4.tick_params(axis='x', rotation=45)

        for bar, score in zip(bars, security_scores):
            if score <= 2:
                bar.set_color('red')
            elif score <= 3:
                bar.set_color('orange')
            else:
                bar.set_color('green')

        plt.tight_layout()
        plt.show()

    def demonstrate_hash_examples(self):
        print("\n=== HASH EXAMPLES ===")
        test_inputs = ["password", "Password", "password1", "cybersecurity2024"]

        for test_input in test_inputs:
            print(f"\nInput: '{test_input}'")
            for algo in ['md5', 'sha256', 'blake2b']:
                hash_val = self.hash_text(test_input, algo)
                print(f"{algo.upper()}: {hash_val}")

def main():
    comparator = HashComparator()
    comparator.run_comprehensive_analysis()
    comparator.demonstrate_hash_examples()

if __name__ == "__main__":
    main()